# 고전 CNN 모델 실습: LeNet-5 / VGG16

이번 노트북에서는 고전적인 CNN 모델 2종을 실습합니다.

1. **LeNet-5** — MNIST 직접 학습 및 평가
2. **VGG16** — ImageNet 사전학습 가중치로 예측 실습

모든 모델의 파라미터 수를 비교하고, 구조를 살펴봅니다.


In [1]:
%matplotlib inline
import os
import logging
import platform
import sys
import numpy as np
import matplotlib.pyplot as plt
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
from tensorflow.keras import layers, models
from pathlib import Path as _Path

tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

if platform.system() == "Windows":
    plt.rc("font", family="Malgun Gothic")
elif platform.system() == "Darwin":
    plt.rc("font", family="AppleGothic")
else:
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

tf.random.set_seed(42)
np.random.seed(42)

_root = next(
    p for p in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]
    if (p / ".cursor" / "runtime_device.py").exists()
)
if str(_root / ".cursor") not in sys.path:
    sys.path.insert(0, str(_root / ".cursor"))
import runtime_device
runtime_device.configure_tensorflow(tf)
print("TensorFlow 버전:", tf.__version__)
from tensorflow.keras import datasets
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
print("NumPy 버전:", np.__version__)


OS: Darwin
학습 장치: MPS (Apple GPU) [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
배치된 장치: /job:localhost/replica:0/task:0/device:GPU:0
TensorFlow 버전: 2.17.1
NumPy 버전: 1.26.4


## [1] LeNet-5 구현 및 MNIST 학습

In [2]:
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print('Train:', x_train.shape, ' Test:', x_test.shape)

Train: (60000, 28, 28, 1)  Test: (10000, 28, 28, 1)


In [3]:
def build_lenet5():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(6, (5,5), activation='tanh', padding='same'),
        layers.AveragePooling2D(pool_size=(2,2)),
        layers.Conv2D(16, (5,5), activation='tanh'),
        layers.AveragePooling2D(pool_size=(2,2)),
        layers.Flatten(),
        layers.Dense(120, activation='tanh'),
        layers.Dense(84, activation='tanh'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

lenet5 = build_lenet5()
lenet5.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
history = lenet5.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=2)

Epoch 1/5


422/422 - 17s - 39ms/step - accuracy: 0.8974 - loss: 0.3569 - val_accuracy: 0.9620 - val_loss: 0.1306


Epoch 2/5


422/422 - 13s - 31ms/step - accuracy: 0.9584 - loss: 0.1368 - val_accuracy: 0.9740 - val_loss: 0.0904


Epoch 3/5


422/422 - 14s - 33ms/step - accuracy: 0.9720 - loss: 0.0910 - val_accuracy: 0.9777 - val_loss: 0.0739


Epoch 4/5


422/422 - 17s - 41ms/step - accuracy: 0.9799 - loss: 0.0662 - val_accuracy: 0.9800 - val_loss: 0.0641


Epoch 5/5


422/422 - 18s - 43ms/step - accuracy: 0.9848 - loss: 0.0509 - val_accuracy: 0.9823 - val_loss: 0.0582


In [5]:
test_loss, test_acc = lenet5.evaluate(x_test, y_test, verbose=0)
print(f"LeNet-5 테스트 정확도: {test_acc:.4f}")

LeNet-5 테스트 정확도: 0.9818


### [생각해보기 - LeNet-5]
1. LeNet-5는 Convolution과 Pooling을 왜 번갈아 쌓나요?
2. 레이어가 출력에 가까워질수록 특징 맵 크기는 줄고 개수는 늘어나는 이유는 무엇인가요?


**나의 답변:**

1. Convolution은 국소 패턴을 뽑고, Pooling은 공간을 줄여 계산량과 과적합을 낮춘다.  
   따라서 둘을 교차하면 해상도는 줄이면서도, 다음 층이 더 넓은 영역을 보게 된다.

2. 앞쪽에서는 엣지처럼 단순한 특징을 조밀하게 보고, 뒤쪽에서는 숫자 전체처럼 추상적인 특징이 필요하다.  
   그래서 공간은 압축하고 채널을 늘려, 정보는 더 압축하면서 표현력은 키운다.  
   이번 실습의 LeNet-5는 28x28x6에서 5x5x16으로 줄어들었고, 테스트 정확도는 0.9818이었다.


## [3] VGG16 (ImageNet 사전학습 가중치)

In [6]:
vgg16 = VGG16(weights='imagenet')
vgg16.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 1000)           │     4,097,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,357,544 (527.79 MB)

 Trainable params: 138,357,544 (527.79 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
img_path = tf.keras.utils.get_file('elephant.jpg', 'https://cdn.hackaday.io/images/1219231493836853483.jpg')
img_vgg = image.load_img(img_path, target_size=(224, 224))
x_vgg = image.img_to_array(img_vgg)
x_vgg = np.expand_dims(x_vgg, axis=0)
x_vgg = preprocess_input(x_vgg)

preds_vgg = vgg16.predict(x_vgg)
print('VGG16 예측 결과:', decode_predictions(preds_vgg, top=3)[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


    0/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

16384/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 6us/step

35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 3us/step


VGG16 예측 결과: [('n02504458', 'African_elephant', 1.0), ('n01871265', 'tusker', 3.00756e-12), ('n02504013', 'Indian_elephant', 6.162328e-15)]


### [생각해보기 - VGG16]
1. VGGNet이 3x3 필터를 깊게 쌓는 이유는 무엇인가요?
2. LeNet-5와 VGG16의 파라미터 수 차이가 큰 까닭은 무엇인가요?


**나의 답변:**

1. 강의에서 VGGNet은 3x3 필터에 stride 1, same padding을 쓴다.  
   그래서 특징 맵 크기를 유지한 채 Convolution을 반복해 층을 깊게 쌓을 수 있다.

2. VGG16은 224x224 컬러 ImageNet을 다루고, 채널이 512까지 늘어나며 마지막에 큰 완전연결층이 있다.  
   반면 LeNet-5는 28x28 흑백 숫자용 얕은 구조라, 파라미터가 61,706개에 머문다.  
   이번 예측에서는 African_elephant 확률이 1.0으로 나왔다.


## [4] 모델별 파라미터 수 비교

In [8]:
models_info = {
    'LeNet-5 (MNIST)': lenet5.count_params(),
    'VGG16 (ImageNet)': vgg16.count_params()
}
for name, params in models_info.items():
    print(f'{name}: {params:,} parameters')

LeNet-5 (MNIST): 61,706 parameters
VGG16 (ImageNet): 138,357,544 parameters


### [생각해보기 - CNN과 파라미터]
1. 같은 MNIST를 MLP에 넣으면 파라미터가 급증하는데, CNN은 왜 훨씬 적은가요?
2. 가중치 공유가 이동 등가성에 어떻게 도움이 되나요?


**나의 답변:**

1. MLP는 픽셀을 한 줄로 펴서 모든 위치에 따로 가중치를 둔다.  
   반면 CNN은 작은 필터를 이미지 전역에 재사용하므로, 파라미터 수가 이미지 크기와 거의 무관하다.  
   그 결과 LeNet-5는 61,706개인데, VGG16은 완전연결층까지 합쳐 138,357,544개로 벌어진다.

2. 같은 필터가 어디서나 같은 엣지를 찾기 때문에, 숫자가 조금 이동해도 비슷한 반응이 나온다.  
   그래서 위치만 바뀐 패턴을 새로 외우지 않아도 된다.
